In [8]:
import numpy as np
import matplotlib.pyplot as plt
import math
import os

#Make an output directory
out_dir = "foucault_plots"
os.makedirs(out_dir, exist_ok=True)

#ODE + RK4

def rhs(t, y, R0):
    """
    Right-hand side of the dimensionless equations:
    x'' = -x + (2/R0) y' + (1/R0^2) x
    y'' = -y - (2/R0) x' + (1/R0^2) y
    y = [x, y_pos, vx, vy]
    """
    x, y_pos, vx, vy = y
    invR = 1.0 / R0
    invR2 = invR**2

    dxdt = vx
    dydt = vy
    dvxdt = -x + 2*invR*vy + invR2*x
    dvydt = -y_pos - 2*invR*vx + invR2*y_pos

    return np.array([dxdt, dydt, dvxdt, dvydt])

def rk4_step(f, t, y, h, R0):
    k1 = f(t, y, R0)
    k2 = f(t + 0.5*h, y + 0.5*h*k1, R0)
    k3 = f(t + 0.5*h, y + 0.5*h*k2, R0)
    k4 = f(t + h, y + h*k3, R0)
    return y + (h/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def integrate(R0, h, t_end):
    """
    Integrate from t=0 to t=t_end with step h.
    Initial conditions: x=1, y=0, vx=0, vy=0.
    """
    n_steps = int(round(t_end / h))
    t_vals = np.linspace(0.0, n_steps*h, n_steps+1)
    y_vals = np.zeros((n_steps+1, 4))

    #initial conditions
    y_vals[0] = np.array([1.0, 0.0, 0.0, 0.0])
    t = 0.0
    for i in range(n_steps):
        y_vals[i+1] = rk4_step(rhs, t, y_vals[i], h, R0)
        t += h

    return t_vals, y_vals

#Trajectories for R0 = 10, 2, 1 

R0_values = [10.0, 2.0, 1.0]
t_end = 20.0 * math.pi      #dimensionless time
h_traj = 0.01               #step size for nice smooth curves

for R0 in R0_values:
    t_vals, y_vals = integrate(R0, h_traj, t_end)
    x = y_vals[:, 0]
    y = y_vals[:, 1]

    plt.figure()
    plt.plot(x, y)
    plt.xlabel("x (dimensionless)")
    plt.ylabel("y (dimensionless)")
    plt.title(f"Pendulum trajectory (R0 = {R0})")
    plt.tight_layout()
    filename = f"trajectory_R0_{str(R0).replace('.', '_')}.png"
    plt.savefig(os.path.join(out_dir, filename), dpi=200)
    plt.close()

#Time series for R0 = 2 

R0 = 2.0
t_vals, y_vals = integrate(R0, h_traj, t_end)
x = y_vals[:, 0]
y = y_vals[:, 1]

plt.figure()
plt.plot(t_vals, x, label="x(t̃)")
plt.plot(t_vals, y, label="y(t̃)")
plt.xlabel("t̃")
plt.ylabel("Position")
plt.title("x(t̃) and y(t̃) for R0 = 2")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "time_series_R0_2.png"), dpi=200)
plt.close()

#Time series for R0 = 10 and R0 = 1

for R0_ts in [10.0, 1.0]:
    t_vals_ts, y_vals_ts = integrate(R0_ts, h_traj, t_end)
    x_ts = y_vals_ts[:, 0]
    y_ts = y_vals_ts[:, 1]

    plt.figure()
    plt.plot(t_vals_ts, x_ts, label="x(t̃)")
    plt.plot(t_vals_ts, y_ts, label="y(t̃)")
    plt.xlabel("t̃")
    plt.ylabel("Position")
    plt.title(f"x(t̃) and y(t̃) for R0 = {R0_ts}")
    plt.legend()
    plt.tight_layout()

    filename = f"time_series_R0_{str(R0_ts).replace('.', '_')}.png"
    plt.savefig(os.path.join(out_dir, filename), dpi=200)
    plt.close()

#Convergence for R0 = 2 

h_values = [0.1, 0.05, 0.025, 0.0125]
t_end_conv = 62.5          #chosen so it's an integer multiple of all h's
sols = []

for h in h_values:
    t_vals_c, y_vals_c = integrate(R0, h, t_end_conv)
    sols.append(y_vals_c[-1])   # final state [x,y,vx,vy]

ref = sols[-1]             #finest step as "reference"

errors_x = [abs(sol[0] - ref[0]) for sol in sols]
errors_y = [abs(sol[1] - ref[1]) for sol in sols]
h_arr = np.array(h_values)

plt.figure()
plt.loglog(h_arr, errors_x, marker="o", label="|x(h) - x_ref|")
plt.loglog(h_arr, errors_y, marker="s", label="|y(h) - y_ref|")
plt.xlabel("Step size h")
plt.ylabel("Error at final time")
plt.title("Self-convergence at t̃ = 62.5 (R0 = 2)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "convergence_R0_2.png"), dpi=200)
plt.close()

#Richardson error bars at final time

p = 4.0                   #RK4 is 4th order
h1, h2 = 0.025, 0.0125
idx1 = h_values.index(h1)
idx2 = h_values.index(h2)
sol_h1 = sols[idx1]
sol_h2 = sols[idx2]

x_R = sol_h2[0] + (sol_h2[0] - sol_h1[0]) / (2**p - 1.0)
y_R = sol_h2[1] + (sol_h2[1] - sol_h1[1]) / (2**p - 1.0)
err_x = abs(x_R - sol_h2[0])
err_y = abs(y_R - sol_h2[1])

print("Richardson numbers (R0 = 2, t̃ = 62.5)")
print("h1 =", h1, "h2 =", h2, "p =", p)
print("sol_h1: x =", sol_h1[0], " y =", sol_h1[1])
print("sol_h2: x =", sol_h2[0], " y =", sol_h2[1])
print("Richardson extrapolated: x_R =", x_R, " y_R =", y_R)
print("Estimated errors: err_x =", err_x, " err_y =", err_y)

plt.figure()
plt.errorbar([0], [sol_h2[0]], yerr=[err_x], fmt="o", capsize=5, label="x")
plt.errorbar([1], [sol_h2[1]], yerr=[err_y], fmt="s", capsize=5, label="y")
plt.xticks([0, 1], ["x", "y"])
plt.ylabel("Value at t̃ = 62.5")
plt.title("Final values with Richardson error bars (R0 = 2)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "final_values_errorbars_R0_2.png"), dpi=200)
plt.close()



Richardson numbers (R0 = 2, t̃ = 62.5)
h1 = 0.025 h2 = 0.0125 p = 4.0
sol_h1: x = 0.959360341567686  y = -0.004505385336298828
sol_h2: x = 0.9593605254275959  y = -0.004505693341386865
Richardson extrapolated: x_R = 0.9593605376849232  y_R = -0.0045057138750594005
Estimated errors: err_x = 1.2257327375309046e-08  err_y = 2.053367253573163e-08
